# ROUTER = the complete routing mechanism
# ROUTER SELECTOR = the decision-maker inside the router

<div align="center">
    <img src="images/routerselector.png" width="500" height="1000">
</div>

| Component                        | What it selects                              | Single or multiple  |
| -------------------------------- | -------------------------------------------- | ------------------- |
| `RouterQueryEngine`              | Query engines                                | Depends on selector |
| `LLMSingleSelector`              | One query engine using text-based LLM output | One                 |
| `PydanticSingleSelector`         | One query engine using function calling      | One                 |
| `LLMMultiSelector`               | Query engines using text-based LLM output    | Multiple            |
| `PydanticMultiSelector`          | Query engines using function calling         | Multiple            |
| `RouterRetriever`                | Retrievers                                   | One or multiple     |
| `ToolRetrieverRouterQueryEngine` | Query-engine tools through retrieval         | One or multiple     |

In [1]:
import logging
import warnings

warnings.filterwarnings("ignore", message="Mixing V1 models and V2 models")
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("llama_index").setLevel(logging.ERROR)
logging.disable(logging.WARNING)

In [2]:
import os
from dotenv import load_dotenv
from llama_index.core import (Settings, SimpleDirectoryReader, VectorStoreIndex)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.tools import (QueryEngineTool, RetrieverTool,)
from llama_index.core.selectors import (LLMSingleSelector, PydanticSingleSelector, LLMMultiSelector, PydanticMultiSelector,)
from llama_index.core.query_engine import (RouterQueryEngine, ToolRetrieverRouterQueryEngine, RetrieverRouterQueryEngine)
from llama_index.core.retrievers import RouterRetriever
from llama_index.core.objects import SimpleToolNodeMapping
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.objects import ObjectIndex

load_dotenv()

# Configure the LLM and embedding model
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Load the three PDF files separately
hope_documents = SimpleDirectoryReader(input_files=["data/HopeAI.pdf"]).load_data()
python_documents = SimpleDirectoryReader(input_files=["data/Python.pdf"]).load_data()
ml_documents = SimpleDirectoryReader(input_files=["data/ML.pdf"]).load_data()

# Create one vector index for each PDF
hope_index = VectorStoreIndex.from_documents(hope_documents)
python_index = VectorStoreIndex.from_documents(python_documents)
ml_index = VectorStoreIndex.from_documents(ml_documents)

# Create query engines
hope_engine = hope_index.as_query_engine(similarity_top_k=3)
python_engine = python_index.as_query_engine(similarity_top_k=3)
ml_engine = ml_index.as_query_engine(similarity_top_k=3)

# Wrap query engines as tools
query_engine_tools = [
    QueryEngineTool.from_defaults(
        query_engine=hope_engine,
        name="hopeai_information",
        description=("Use this tool for questions about Hope AI, its courses, services, training programs, and placement support.")
    ),
    QueryEngineTool.from_defaults(
        query_engine=python_engine,
        name="python_information",
        description=("Use this tool for questions about Python programming, syntax, variables, functions, classes, and libraries.")
    ),
    QueryEngineTool.from_defaults(
        query_engine=ml_engine,
        name="machine_learning_information",
        description=("Use this tool for questions about machine learning, algorithms, training, classification, and regression.")
    )
]

<h1 style="color: #16A34A;">1. ROUTER QUERY ENGINE</h1>

# Routes between query engines.

<div align="center">
    <img src="images/routerqueryengine.png" width="300" height="700">
</div>

In [3]:
router_engine = RouterQueryEngine.from_defaults(
    query_engine_tools=query_engine_tools,
    select_multi=False,
    verbose=True
)
response = router_engine.query("What courses are offered by Hope AI?")
print("\nAnswer:")
print(response)

Selecting query engine 0: The question is about the courses offered by Hope AI, which directly relates to the information provided in choice 1..

Answer:
Hope AI offers courses in Artificial Intelligence and Data Science, covering a wide range of topics including Python Basics, various Python libraries (Numpy, Pandas, Matplotlib, Sklearn, Seaborn), data science techniques, machine learning (regression, classification, clustering), advanced machine learning, artificial neural networks, deep learning, computer vision, time series analysis, natural language processing, and web development with Django and AI model integration. Additionally, there are capstone portfolio projects that focus on real-time scenario-based applications.


<h1 style="color: #16A34A;">2. LLM SINGLE SELECTOR</h1>

# LLMSingleSelector uses the LLM to select exactly one query engine.

<div align="center">
    <img src="images/llmsingleselector.png" width="300" height="700">
</div>

In [4]:
llm_single_selector = LLMSingleSelector.from_defaults(llm=Settings.llm)
llm_single_router = RouterQueryEngine(
    selector=llm_single_selector,
    query_engine_tools=query_engine_tools,
    verbose=True
)
response = llm_single_router.query("Explain Python functions.")
print("\nAnswer:")
print(response)

Selecting query engine 1: The question specifically asks about Python functions, which falls under the category of Python programming and its related concepts..

Answer:
A function in Python groups reusable instructions, allowing for organized and modular code. It can accept inputs through parameters and send a result back to the caller using the return statement. This structure enables code to be reused without rewriting the same instructions multiple times.


<h1 style="color: #16A34A;">3. PYDANTIC SINGLE SELECTOR</h1>

# PydanticSingleSelector uses structured function calling to select exactly one tool.

<div align="center">
    <img src="images/pydanticsingleselector.png" width="300" height="700">
</div>

In [5]:
pydantic_single_selector = PydanticSingleSelector.from_defaults(llm=Settings.llm)
pydantic_single_router = RouterQueryEngine(
    selector=pydantic_single_selector,
    query_engine_tools=query_engine_tools,
    verbose=True
)
response = pydantic_single_router.query("What is supervised learning?")
print("\nAnswer:")
print(response)

Selecting query engine 2: The question 'What is supervised learning?' pertains to machine learning concepts, specifically related to training and classification..

Answer:
Supervised learning is a type of machine learning where the model learns from examples that have known targets. It involves predicting categories in classification tasks, such as determining whether an email is spam or not, or predicting numerical values in regression tasks, such as estimating house prices.


<h1 style="color: #16A34A;">4. LLM MULTI SELECTOR</h1>

# LLMMultiSelector can select multiple query engines. It is useful when the answer requires information from more than one PDF.

<div align="center">
    <img src="images/llmmultiselector.png" width="500" height="700">
</div>

In [6]:
llm_multi_selector = LLMMultiSelector.from_defaults(llm=Settings.llm)
llm_multi_router = RouterQueryEngine(
    selector=llm_multi_selector,
    query_engine_tools=query_engine_tools,
    verbose=True
)
response = llm_multi_router.query("What does Hope AI teach about Python and machine learning?")
print("\nCombined Answer:")
print(response)

Selecting query engine 0: Hope AI is mentioned, indicating relevance to its courses and services..
Selecting query engine 1: This choice specifically addresses questions about Python programming, which is part of the inquiry..
Selecting query engine 2: This choice addresses questions about machine learning, which is also part of the inquiry..

Combined Answer:
Hope AI teaches a comprehensive curriculum that covers both Python and machine learning. In Python, the program emphasizes foundational concepts such as control statements, functions, lists, and tuples, along with essential libraries like Numpy, Pandas, Matplotlib, Sklearn, and Seaborn. It also includes data science fundamentals, focusing on loading datasets, data cleaning, and feature scaling.

In the realm of machine learning, the curriculum encompasses various techniques, including regression methods (like simple and multiple linear regression, decision trees, and random forest), classification methods (such as logistic regres

<h1 style="color: #16A34A;">5. PYDANTIC MULTI SELECTOR</h1>

# PydanticMultiSelector uses structured function calling and can select multiple tools.

<div align="center">
    <img src="images/pydanticmultiselector.png" width="500" height="700">
</div>

In [7]:
pydantic_multi_selector = PydanticMultiSelector.from_defaults(llm=Settings.llm)
pydantic_multi_router = RouterQueryEngine(
    selector=pydantic_multi_selector,
    query_engine_tools=query_engine_tools,
    verbose=True
)
response = pydantic_multi_router.query("Compare the Python concepts and machine learning concepts taught by Hope AI.")
print("\nCombined Answer:")
print(response)

Selecting query engine 0: The question involves comparing concepts taught by Hope AI, which relates to its courses and training programs..
Selecting query engine 1: The question specifically mentions Python concepts, which are covered in the second choice..
Selecting query engine 2: The question also involves machine learning concepts, which are covered in the third choice..

Combined Answer:
The Python concepts encompass foundational programming elements such as control statements, functions, lists, tuples, and the use of libraries like Numpy, Pandas, Matplotlib, Sklearn, and Seaborn. This section focuses on essential skills for programming and data manipulation, including data loading, cleaning, and feature scaling.

In contrast, the machine learning concepts cover a range of techniques and methodologies. This includes regression methods (like simple and multiple linear regression, decision trees, and random forests), classification techniques (such as logistic regression, SVM, Naive

<h1 style="color: #16A34A;">6. ROUTER RETRIEVER</h1>

# RouterRetriever routes the question between retrievers instead of query engines. The retrieved nodes are then given to a query engine.

<div align="center">
    <img src="images/routerretriever_selector.png" width="300" height="700">
</div>

In [9]:
# Create retrievers from the three indexes
hope_retriever = hope_index.as_retriever(similarity_top_k=3)
python_retriever = python_index.as_retriever(similarity_top_k=3)
ml_retriever = ml_index.as_retriever(similarity_top_k=3)

# Wrap retrievers as tools
retriever_tools = [
    RetrieverTool.from_defaults(
        retriever=hope_retriever,
        name="hopeai_retriever",
        description="Retrieves information about Hope AI."
    ),
    RetrieverTool.from_defaults(
        retriever=python_retriever,
        name="python_retriever",
        description="Retrieves information about Python programming."
    ),
    RetrieverTool.from_defaults(
        retriever=ml_retriever,
        name="ml_retriever",
        description="Retrieves information about machine learning."
    )
]

router_retriever = RouterRetriever(selector=PydanticSingleSelector.from_defaults(llm=Settings.llm),
    retriever_tools=retriever_tools,
    llm=Settings.llm,
    verbose=True
)
router_retriever_engine = RetrieverQueryEngine.from_args(retriever=router_retriever, llm=Settings.llm)
response = router_retriever_engine.query("What is a Python class?")
print("\nAnswer:")
print(response)


Answer:
A Python class is a blueprint for creating objects that encapsulate data and functionality. It allows for the definition of attributes (data) and methods (functions) that operate on the data, enabling the creation of complex data structures and behaviors.


<h1 style="color: #16A34A;">7. TOOL RETRIEVER ROUTER QUERY ENGINE</h1>

# This router first performs vector retrieval over the tool descriptions. It is useful when there are many query-engine tools.

<div align="center">
    <img src="images/toolretriever.png" width="300" height="700">
</div>

In [10]:
# Map tools to nodes
tool_mapping = SimpleToolNodeMapping.from_objects(query_engine_tools)

# Build an index over the query-engine tools
tool_index = ObjectIndex.from_objects(
    query_engine_tools,
    tool_mapping,
    VectorStoreIndex
)

# Retrieve the two most relevant tools
tool_retriever = tool_index.as_retriever(similarity_top_k=2)

# Create tool-retriever router
tool_retriever_router = ToolRetrieverRouterQueryEngine(
    retriever=tool_retriever,
    llm=Settings.llm
)

response = tool_retriever_router.query("Explain Python and machine learning courses available at Hope AI.")
retrieved_tools = response.metadata.get("retrieved_tools", [])
print("\nRetrieved tools:")
for tool in retrieved_tools:
    print("-", tool.metadata.name)
print("\nAnswer:")
print(response)


Retrieved tools:
- hopeai_information
- machine_learning_information

Answer:
The courses at Hope AI cover a wide range of topics in Python and machine learning. The Python course focuses on fundamental concepts such as control statements, functions, lists, and tuples. It also introduces important libraries like Numpy, Pandas, Matplotlib, Sklearn, and Seaborn, along with foundational data science skills, including loading datasets, data cleaning, and feature scaling.

In the machine learning segment, the curriculum encompasses various techniques, including regression methods (simple and multiple linear regression, decision trees, and random forests), classification techniques (logistic regression, SVM, Naive Bayes, and KNN), and clustering methods (K-Means and hierarchical clustering). Additionally, advanced topics are covered, focusing on feature selection and dimensionality reduction techniques such as PCA and LDA.

Overall, these courses aim to equip students with both theoretical 